In [1]:
import os
import pickle
import sys
sys.path.append(os.path.join(os.getcwd(), "metient/jupyter_notebooks/lineage_tracing"))
import lineage_tracing_utilities as lt
import torch
import numpy as np
import gzip
import networkx as nx
from ete3 import Tree
import metient.util.vertex_labeling_util as vutil

import metient as met
output_dir = "/data/morrisq/divyak/projects/metient/metient/jupyter_notebooks/lineage_tracing/outputs"

In [5]:


# Convert networkx tree to ete3 tree
def networkx_to_ete(tree, root):
    # Recursively build the tree
    ete_tree = Tree(name=str(root))
    for child in tree.successors(root):
        ete_tree.add_child(networkx_to_ete(tree, child))
    return ete_tree


def nodes_to_white(tree, reference_V):
    nodes = []
    for node in tree.nodes():
        if np.argmax(V[:, node]) != np.argmax(reference_V[:, node]):
            nodes.append(node)
    return nodes

lt_dir = "/data/morrisq/divyak/projects/metient/metient/data/quinn_lt_2021"
metient_results_dir = os.path.join(lt_dir, "metient_outputs", "2048bs_200runs_05142025_r1")

CLONE = 15

with gzip.open(os.path.join(metient_results_dir,f"{CLONE}_LL.pkl.gz") ,"rb") as f:
    pckl = pickle.load(f)
    
num_solutions = len(pckl['clone_tree_labeling_matrices'])
print(f"{num_solutions} solutions")
ordered_sites = pckl['ordered_anatomical_sites']
losses = pckl['losses']
A = met.adjacency_matrix_from_parents(pckl['full_adjacency_matrices'][0])

root_idx = vutil.get_root_index(A)
# This is the solution to get the diff of labels for
reference_idx = num_solutions - 1
reference_V = pckl['clone_tree_labeling_matrices'][reference_idx]

for tree_idx in range(len(pckl['clone_tree_labeling_matrices'])):
    print("Solution",tree_idx)
    V = pckl['clone_tree_labeling_matrices'][tree_idx]
    
    tree = nx.DiGraph()
    edges = zip(A._indices()[0].tolist(), A._indices()[1].tolist())
    tree.add_edges_from(edges)
    num_nodes = V.shape[1]
    tree.add_nodes_from(range(num_nodes))

    # If this is not the first, best solution, we want to label all nodes that 
    # have the same label as the best migration history "white"
    diff_label_from_ref_soln_nodes = nodes_to_white(tree, reference_V) if tree_idx != reference_idx else range(num_nodes)
    print(len(diff_label_from_ref_soln_nodes))
    for node in tree.nodes():

        tissue = ordered_sites[np.argmax(V[:, node])]
        if node not in diff_label_from_ref_soln_nodes:
            tissue = "W"
        tree.nodes[node]['tissue'] = tissue
    
    ete_tree = networkx_to_ete(tree, root_idx)
    # Add NHX-style metadata (assuming one type of metadata here, for simplicity)
    for node in ete_tree.traverse():
        if int(node.name) in tree.nodes:
            label = tree.nodes[int(node.name)]['tissue']
            node.add_features(NHX=label)

    # Export to Newick format
    newick_str = ete_tree.write(features=["NHX"])
    with open(os.path.join(lt_dir, "newick_outputs",f'{CLONE}_solution{tree_idx}_tree_with_nhx_W.newick'), 'w') as f:
        f.write(newick_str)
        f.flush()

    

23 solutions
Solution 0
129
Solution 1
131
Solution 2
37
Solution 3
29
Solution 4
40
Solution 5
35
Solution 6
28
Solution 7
30
Solution 8
26
Solution 9
28
Solution 10
30
Solution 11
18
Solution 12
17
Solution 13
20
Solution 14
20
Solution 15
18
Solution 16
16
Solution 17
12
Solution 18
9
Solution 19
11
Solution 20
6
Solution 21
5
Solution 22
1122
